In [2]:
import pandas as pd
import numpy as np

# Cibler le fichier à traiter en fonction de l'année :
annee_extraction = "2025"
cible = f"donnees_meteo_{annee_extraction}_COMPLET.csv"

# Chargement des données :
df = pd.read_csv(cible, sep=';')

# Création de la variable cible (target).
# L'objectif du modèle étant de prédire la pluie, on transforme la variable # "pluie_1h_mm" en variable binaire :
    # 1 = il pleut, 0 = il ne pleut pas.
df['pleut_il'] = (df['pluie_1h_mm'] > 0).astype(int)

# Stockage du nom de la variable cible pour faciliter son utilisation :
target = 'pleut_il'

# Affichage d'informations globales sur le dataset (petit contrôle) :
print(f"--- Analyse de pertinence par STATION ---\n")
print(f"Nombre total de lignes : {len(df)}")
print(f"Nombre de stations uniques : {df['station_id'].nunique()}\n")

# Récupération de la liste des stations.
# Cette liste permettra d'analyser les données station par station.
stations = df['station_id'].unique()

# Liste qui va contenir les résultats d'analyse pour toutes les stations.
all_stats = []

# Boucle : analyse station par station.
# Cette approche permet d'identifier d'éventuelles différences de qualité entre les capteurs :
for station in stations:
    # Filtrage du dataset pour ne conserver qu'une station :
    df_station = df[df['station_id'] == station].copy()

    # stocker les métriques de la station courante :
    stats = []

     # Analyse colonne par colonne :
    for col in df_station.columns:
        # On ignore certaines colonnes non pertinentes pour l'analyse :
            # dh_utc : variable temporelle brute.
            # station_id : identifiant technique.
            # pleut_il : variable cible (ne doit pas être utilisée comme feature).
        if col in ['dh_utc', 'station_id', 'pleut_il']:
            continue

        # Nombre total d'observations pour la station :
        total = len(df_station)

        # Calcul du nombre de valeurs manquantes.
        # Les données manquantes peuvent impacter fortement les performances du model :
        missing = df_station[col].isnull().sum()

        # Calcul du pourcentage de valeurs manquantes :
        missing_pct = (missing / total) * 100

        # Calcul du nombre de valeurs uniques :
        unique_vals = df_station[col].nunique()

        # Initialisation de la corrélation :
        correlation = np.nan
        if pd.api.types.is_numeric_dtype(df_station[col]):
            # Calcul de la corrélation uniquement s'il y a assez de données non-null :
            if df_station[col].notnull().sum() > 1:

                # Calcul de la corrélation avec la variable cible :
                correlation = df_station[col].corr(df_station[target])

        # Ajout des résultats dans une liste :
        stats.append({
            'station': station,
            'colonne': col,
            'manquantes': missing,
            'pourcentage_manquant': f"{missing_pct:.2f}%",
            'correlation_avec_pluie': f"{correlation:.3f}" if not np.isnan(correlation) else "N/A"
        })
    
    # Ajout des stats de cette station à la liste globale :
    all_stats.extend(stats)

# Création d'un DataFrame final contenant toutes les analyses :
df_final_stats = pd.DataFrame(all_stats)

# Affichage complet pour afficher tout le tableau dans la console :
pd.set_option('display.max_rows', None) # Affiche toutes les lignes.
pd.set_option('display.width', 1000)    # Largeur suffisante.
print(df_final_stats.to_string(index=False))




--- Analyse de pertinence par STATION ---

Nombre total de lignes : 172497
Nombre de stations uniques : 4

station             colonne  manquantes pourcentage_manquant correlation_avec_pluie
  000BV    temperature_degC           8                0.02%                    N/A
  000BV        pression_hPa           0                0.00%                    N/A
  000BV          humidite_%           8                0.02%                    N/A
  000BV point_de_rosee_degC           8                0.02%                    N/A
  000BV     vent_moyen_km/h          26                0.05%                    N/A
  000BV   vent_rafales_km/h       51675              100.00%                    N/A
  000BV  vent_direction_deg           8                0.02%                    N/A
  000BV         pluie_3h_mm       51675              100.00%                    N/A
  000BV         pluie_1h_mm       51675              100.00%                    N/A
  000CT    temperature_degC           0              

C:\Users\ATOMIQ\PyCharmMiscProject\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\ATOMIQ\PyCharmMiscProject\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
